# Iterative Cleaning with an Autoencoder
## Train on digits 0-8; inject digit 9

Compare a no-cleaning baseline, iterative removal of suspicious images, and a no-anomaly ceiling.

- Normal digits: 0-8
- Injected digit: 9
- Digit 9 resembles normal digits 4 and 7.
- Labels are used for evaluation, not for training or removal.
- The score combines latent distance and reconstruction error, as set by `SCORE_MODE`.

| Method | Description |
|---|---|
| No cleaning | Train and evaluate the initial contaminated set without removing images. |
| Iterative cleaning | Remove suspicious images over several rounds, retraining between rounds. |
| Ceiling | Train on normal images only, with no injected anomalies. |

Evaluation uses only known-normal training examples as kNN references; that reference is an oracle for evaluation, not a cleaning signal available in deployment. The baseline uses the existing round-0 model. Run the cells in order.

## 1. Imports

In [ ]:
import math                                   # math helpers (ceil etc.)
import random                                 # Python's random number generator (seeded for repeatability)
import time                                   # to time the run
from datetime import datetime                 # unique names for saved runs
from pathlib import Path                      # clean way to build file/folder paths

import numpy as np                            # arrays and numerical helpers
import torch                                  # PyTorch: tensors and neural networks
import torch.nn as nn                         # neural-network layers
import torch.nn.functional as F               # functions like mse_loss, one_hot, normalize
import matplotlib.pyplot as plt               # plotting
from sklearn.metrics import roc_auc_score, average_precision_score   # ranking metrics (explained below)
from torchvision import datasets              # downloads MNIST for us

## 2. Configuration
Change the scenario, data size, training, scoring, and output settings here.

In [ ]:
# ---------- WHICH DIGITS ARE NORMAL / ANOMALOUS ----------
SCENARIO_TAG = "Train 0-8 inject 9"
NORMAL_DIGITS = [0, 1, 2, 3, 4, 5, 6, 7, 8]
ANOMALY_DIGITS = [9]

# ---------- DATA SIZE ----------
POOL_SIZE = 20000
INJECT_FRACTION = 0.05

# ---------- CLEANING SETTINGS ----------
CLEANING_ROUNDS = 3
REMOVE_FRACTION = 0.02
SCORE_MODE = "combined"
NUM_LATENT_SPACES = 5
NUM_SEEDS = NUM_LATENT_SPACES

# ---------- AUTOENCODER SETTINGS ----------
LATENT_DIM = 64
EPOCHS = 15
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
K_NEIGHBORS = 5

# ---------- OUTPUT ----------
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = (Path("Iterations") / SCENARIO_TAG / f"{NUM_LATENT_SPACES} Spaces"
              / f"{LATENT_DIM} Latent Dimensions")
RESULT_STEM = f"{SCENARIO_TAG} ({LATENT_DIM} Latent Dimensions) {RUN_ID}"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"
print(f"Model and dataset device: {DEVICE}")
if USE_AMP:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

assert not set(NORMAL_DIGITS) & set(ANOMALY_DIGITS), "digit lists must not overlap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE} | normal {NORMAL_DIGITS} | anomalies {ANOMALY_DIGITS}")
if USE_AMP:
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)} | mixed precision enabled")
print(f"Saving this run to: {OUTPUT_DIR} (run {RUN_ID})")

LOG_LINES = []
def log(text=""):
    """Print a line and remember it for the saved results file."""
    print(text)
    LOG_LINES.append(text)

## 3. Autoencoder
The encoder maps each image to a `LATENT_DIM` vector; the decoder reconstructs it.

In [ ]:
class ConvAutoencoder(nn.Module):
    """Convolutional autoencoder for 28x28 grayscale images."""

    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_autoencoder(images):
    """Train a fresh autoencoder, keeping the current training subset on the selected device."""
    model = ConvAutoencoder(LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    images = images.to(DEVICE)
    n = images.size(0)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    model.train()
    for _ in range(EPOCHS):
        permutation = torch.randperm(n, device=DEVICE)
        for start in range(0, n, BATCH_SIZE):
            batch = images[permutation[start:start + BATCH_SIZE]]
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                reconstruction, _ = model(batch)
            loss = F.mse_loss(reconstruction.float(), batch.float())
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
    return model

## 4. Cleaning signals

- Latent kNN distance: average distance to nearby training embeddings.
- Reconstruction error: squared pixel error divided by image ink.

The signals are ranked and combined according to `SCORE_MODE`.

In [ ]:
@torch.no_grad()
def encode(model, images, batch_size=2048):
    """Encode images in chunks and return full-precision latent vectors on the CPU."""
    model.eval()
    encoded = []
    for start in range(0, images.size(0), batch_size):
        batch = images[start:start + batch_size].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            embeddings = model.encoder(batch)
        encoded.append(embeddings.float().cpu())
    return torch.cat(encoded)


@torch.no_grad()
def recon_error_per_ink(model, images, batch_size=2048):
    """Squared reconstruction error divided by each image's total ink."""
    model.eval()
    out = []
    for start in range(0, images.size(0), batch_size):
        batch = images[start:start + batch_size].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            reconstruction, _ = model(batch)
        reconstruction = reconstruction.float()
        squared_error = ((reconstruction - batch.float()) ** 2).sum(dim=(1, 2, 3))
        ink = batch.float().sum(dim=(1, 2, 3)) + 1e-6
        out.append((squared_error / ink).cpu())
    return torch.cat(out).numpy()


def knn_distance(query, reference, k, exclude_self=False):
    """Mean Euclidean distance to the k nearest reference embeddings."""
    query_device = query.to(DEVICE)
    reference_device = reference.to(DEVICE)
    out = []
    for start in range(0, query_device.size(0), 1000):
        distances = torch.cdist(query_device[start:start + 1000], reference_device)
        if exclude_self:
            nearest = distances.topk(k + 1, dim=1, largest=False).values[:, 1:]
        else:
            nearest = distances.topk(k, dim=1, largest=False).values
        out.append(nearest.mean(dim=1).cpu())
    return torch.cat(out).numpy()


def to_rank(values):
    """Convert scores to ranks in [0, 1]."""
    return np.argsort(np.argsort(values)) / (len(values) - 1)


def pick_score(knn, recon):
    """Combine the selected suspicion signals into one score."""
    if SCORE_MODE == "latent":
        return to_rank(knn)
    if SCORE_MODE == "recon":
        return to_rank(recon)
    return (to_rank(knn) + to_rank(recon)) / 2


def probe_predict(ref_lat, ref_labels, query_lat, k):
    """Classify query embeddings by majority vote among their nearest labelled references."""
    ref_lat_device = ref_lat.to(DEVICE)
    query_lat_device = query_lat.to(DEVICE)
    ref_labels_t = torch.from_numpy(ref_labels).long().to(DEVICE)
    preds = []
    for start in range(0, query_lat_device.size(0), 1000):
        distances = torch.cdist(query_lat_device[start:start + 1000], ref_lat_device)
        indices = distances.topk(k, dim=1, largest=False).indices
        votes = F.one_hot(ref_labels_t[indices], 10).sum(dim=1)
        preds.append(votes.argmax(dim=1).cpu())
    return torch.cat(preds).numpy()

## 5. Data
Load MNIST, select the configured normal and injected digits, and build fixed test sets. The loader checks this folder and its parents for `data/MNIST`.

In [ ]:
def evaluate(model, train_imgs, train_labels):
    """Grade using only known-normal training examples as kNN references."""
    train_lat = encode(model, train_imgs)
    normal_reference_mask = np.isin(train_labels, NORMAL_DIGITS)
    reference_lat = train_lat[normal_reference_mask]
    reference_labels = train_labels[normal_reference_mask]
    if len(reference_labels) < K_NEIGHBORS:
        raise ValueError("Not enough normal training examples for the kNN evaluation reference.")

    normal_lat = encode(model, ctx["normal_x"])
    preds = probe_predict(reference_lat, reference_labels, normal_lat, K_NEIGHBORS)
    acc = float((preds == ctx["normal_y"]).mean())

    det_lat = encode(model, ctx["det_x"])
    knn = knn_distance(det_lat, reference_lat, K_NEIGHBORS)
    rec = recon_error_per_ink(model, ctx["det_x"])
    score = pick_score(knn, rec)
    y = ctx["det_is_anom"]
    return {
        "acc": acc,
        "auc": roc_auc_score(y, score),
        "ap": average_precision_score(y, score),
        "auc_latent": roc_auc_score(y, knn),
        "auc_recon": roc_auc_score(y, rec),
        "score": score,
    }

## 6. Evaluation
Accuracy uses a kNN classifier on normal test digits. AUROC and average precision use the selected novelty score on normal and anomaly test images. Both use a known-normal training subset as their kNN reference; this is an evaluation oracle, not a cleaning signal available in deployment.

In [ ]:
def find_mnist_data_root():
    """Find MNIST under the project root even when the kernel starts in a subfolder."""
    working_dir = Path.cwd().resolve()
    required_raw_files = (
        "train-images-idx3-ubyte",
        "train-labels-idx1-ubyte",
        "t10k-images-idx3-ubyte",
        "t10k-labels-idx1-ubyte",
    )
    for base_dir in (working_dir, *working_dir.parents):
        data_root = base_dir / "data"
        mnist_dir = data_root / "MNIST"
        raw_dir = mnist_dir / "raw"
        processed_dir = mnist_dir / "processed"
        has_processed_data = (processed_dir / "training.pt").is_file() and (processed_dir / "test.pt").is_file()
        has_raw_data = raw_dir.is_dir() and all(
            (raw_dir / name).is_file() or (raw_dir / f"{name}.gz").is_file()
            for name in required_raw_files
        )
        if has_processed_data or has_raw_data:
            return data_root
    return working_dir / "data"


DATA_ROOT = find_mnist_data_root()
print(f"MNIST data root: {DATA_ROOT}")
train_set = datasets.MNIST(DATA_ROOT, train=True, download=True)
test_set = datasets.MNIST(DATA_ROOT, train=False, download=True)

train_x = train_set.data.float().div(255).unsqueeze(1).to(DEVICE)
train_y = train_set.targets.numpy()
test_x = test_set.data.float().div(255).unsqueeze(1).to(DEVICE)
test_y = test_set.targets.numpy()

normal_train_idx = np.where(np.isin(train_y, NORMAL_DIGITS))[0]
anomaly_train_idx = np.where(np.isin(train_y, ANOMALY_DIGITS))[0]
POOL_USED = min(POOL_SIZE, len(normal_train_idx))
N_INJECT = int(POOL_USED * INJECT_FRACTION / (1 - INJECT_FRACTION))
assert N_INJECT <= len(anomaly_train_idx), "not enough anomaly images to inject"

normal_mask = np.isin(test_y, NORMAL_DIGITS)
anomaly_mask = np.isin(test_y, ANOMALY_DIGITS)
normal_mask_device = torch.as_tensor(normal_mask, dtype=torch.bool, device=DEVICE)
anomaly_mask_device = torch.as_tensor(anomaly_mask, dtype=torch.bool, device=DEVICE)
ctx = {
    "normal_x": test_x[normal_mask_device],
    "normal_y": test_y[normal_mask],
    "det_x": torch.cat([test_x[normal_mask_device], test_x[anomaly_mask_device]]),
    "det_is_anom": np.concatenate([np.zeros(normal_mask.sum(), int),
                                   np.ones(anomaly_mask.sum(), int)]),
}

print(f"Normal training images available: {len(normal_train_idx)} | using {POOL_USED}")
print(f"Anomaly images injected per run: {N_INJECT} "
      f"({N_INJECT / (POOL_USED + N_INJECT) * 100:.1f}% of the training set)")
print(f"Test: {len(ctx['normal_y'])} normal images, {int(ctx['det_is_anom'].sum())} anomaly images")

## 7. Experiment
For each seed, compare the round-0 no-cleaning baseline and final iterative cleaning with a no-anomaly ceiling. The ceiling uses only normal training images and represents a best-case reference.

In [ ]:
def run_seed(seed):
    rng = np.random.RandomState(seed)                                  # this seed's own random generator

    # ---- build the training set: real normal images + injected anomalies ----
    pool_idx = rng.choice(normal_train_idx, POOL_USED, replace=False)  # random normal images (no repeats)
    inj_idx = rng.choice(anomaly_train_idx, N_INJECT, replace=False)   # random anomaly images to inject
    all_idx = np.concatenate([pool_idx, inj_idx])
    all_x = train_x[torch.as_tensor(all_idx, dtype=torch.long, device=DEVICE)]  # keep the run's images on the selected device
    all_y = train_y[all_idx]                                           # their TRUE labels (grading only)
    is_inj = np.concatenate([np.zeros(len(pool_idx), bool),            # hidden ground truth: which are injected
                             np.ones(len(inj_idx), bool)])             # (grading only; the algorithm never sees it)
    n0 = len(all_x)
    per_round = int(n0 * REMOVE_FRACTION)                              # images removed each round
    log(f"\n===== seed {seed + 1}/{NUM_SEEDS} | {len(pool_idx)} normal + {N_INJECT} injected "
        f"| removing {per_round} per round for {CLEANING_ROUNDS} rounds =====")

    def select_images(indices):
        return all_x[torch.as_tensor(indices, dtype=torch.long, device=DEVICE)]

    def fit_model(indices, offset):
        set_seed(seed * 100 + offset)
        return train_autoencoder(select_images(indices))

    def training_signals(model, indices):
        """Both suspicion signals for the images currently in the training set."""
        images = select_images(indices)
        lat = encode(model, images)
        knn = knn_distance(lat, lat, K_NEIGHBORS, exclude_self=True)   # self excluded: a point can't be its own neighbour
        rec = recon_error_per_ink(model, images)
        return knn, rec

    out = {"n0": n0, "n_inj": N_INJECT}
    current = np.arange(n0)                                            # indices of images still in the training set
    hits_per_round = []                                                # injected images removed in each round

    # ---- iterative cleaning loop ----
    for r in range(CLEANING_ROUNDS + 1):
        model = fit_model(current, r)
        if r == 0:
            baseline_images = select_images(current)
            out["baseline"] = evaluate(model, baseline_images, all_y[current])
            log(f"  [no-cleaning] n={len(current):>6} | accuracy {out['baseline']['acc'] * 100:5.2f}% "
                f"| detection AUROC {out['baseline']['auc'] * 100:5.2f}%")
        if r == CLEANING_ROUNDS:                                       # final round: just grade, don't remove more
            final_images = select_images(current)
            out["iterative"] = evaluate(model, final_images, all_y[current])
            log(f"  [iterative] n={len(current):>6} | accuracy {out['iterative']['acc'] * 100:5.2f}% "
                f"| detection AUROC {out['iterative']['auc'] * 100:5.2f}%")
            out["inj_left"] = int(is_inj[current].sum()) / len(current)
            break

        knn, rec = training_signals(model, current)
        scores = pick_score(knn, rec)                                  # suspicion score per training image
        worst = np.argsort(scores)[-per_round:]                        # positions of the highest-scoring (most suspicious) images
        removed = current[worst]                                       # their indices in all_x
        hits = int(is_inj[removed].sum())                              # how many were truly injected (grading only)
        hits_per_round.append(hits)
        log(f"      removed {per_round} | {hits} injected (precision {hits / per_round * 100:.1f}%, "
            f"chance would be {N_INJECT / n0 * 100:.1f}%)")
        if seed == 0:
            out.setdefault("selected_anomalies", [])
            removed_images = all_x[torch.as_tensor(removed, dtype=torch.long, device=DEVICE), 0].detach().cpu().numpy()
            out["selected_anomalies"].extend(
                {"image": image, "digit": int(all_y[idx]), "injected": bool(is_inj[idx]),
                 "score": float(scores[position]), "round": r + 1, "train_index": int(all_idx[idx])}
                for position, idx, image in zip(worst, removed, removed_images)
            )
        current = np.delete(current, worst)                            # drop them from the training set

    out["hits_per_round"] = hits_per_round

    # ---- ceiling: train on normal images only (no anomalies at all) ----
    ceiling_indices = np.arange(len(pool_idx))                          # first entries are the normal images
    ceiling_model = fit_model(ceiling_indices, 92)
    ceiling_images = select_images(ceiling_indices)
    out["ceiling"] = evaluate(ceiling_model, ceiling_images, all_y[ceiling_indices])
    log(f"  [ceiling  ] n={len(ceiling_indices):>6} | accuracy {out['ceiling']['acc'] * 100:5.2f}% "
        f"| detection AUROC {out['ceiling']['auc'] * 100:5.2f}%")
    return out


## 8. Run
This fits one model per cleaning pass plus one no-anomaly ceiling model for each seed. The round-0 model also provides the no-cleaning baseline without another fit. `NUM_LATENT_SPACES` controls the seed count.

In [ ]:
start_time = time.time()
log("=" * 70)
log(f"SCENARIO: Train on digits 0-8, inject 9  (latent dim {LATENT_DIM}, score mode {SCORE_MODE})")
log("=" * 70)

results = [run_seed(s) for s in range(NUM_SEEDS)]                      # one full experiment per seed

log(f"\n(finished in {(time.time() - start_time) / 60:.1f} minutes)")


## 9. Summary
Results compare no cleaning, iterative cleaning, and the no-anomaly ceiling across seeds, including latent-distance and reconstruction AUROC.

In [ ]:
def mean_std(values):
    v = np.array(values) * 100                                         # to percent
    return f"{v.mean():6.2f} +/- {v.std():4.2f}"

methods = [("No cleaning", "baseline"), ("Iterative cleaning", "iterative"),
           ("Ceiling (no anomalies)", "ceiling")]

log("=" * 70)
log("RESULTS: Train on digits 0-8, inject 9")
log("=" * 70)
log(f"{'Method':<24} {'Accuracy %':>16} {'Detection AUROC %':>20} {'Avg precision %':>18} "
    f"{'Latent AUROC %':>16} {'Recon AUROC %':>16}")
for name, key in methods:
    log(f"{name:<24} {mean_std([r[key]['acc'] for r in results]):>16} "
        f"{mean_std([r[key]['auc'] for r in results]):>20} "
        f"{mean_std([r[key]['ap'] for r in results]):>18} "
        f"{mean_std([r[key]['auc_latent'] for r in results]):>16} "
        f"{mean_std([r[key]['auc_recon'] for r in results]):>16}")

for reference, label in [("baseline", "no-cleaning baseline"), ("ceiling", "no-anomaly ceiling")]:
    log(f"\nPer-seed difference in percentage points (iterative cleaning minus {label}):")
    for metric, metric_label in [("acc", "Accuracy"), ("auc", "Detection AUROC"),
                                 ("ap", "Average precision"), ("auc_latent", "Latent AUROC"),
                                 ("auc_recon", "Reconstruction AUROC")]:
        differences = np.array([r["iterative"][metric] - r[reference][metric] for r in results]) * 100
        log(f"  {metric_label:<24} {differences.mean():+.2f} +/- {differences.std():.2f}")

# ---- how well did the removal step find the injected images? ----
n0, n_inj = results[0]["n0"], results[0]["n_inj"]
log("\nREMOVAL QUALITY (did the suspicion score find the injected images?)")
prec = np.mean([r["hits_per_round"] for r in results], axis=0)         # average injected-hits per round across seeds
per_round = int(n0 * REMOVE_FRACTION)
log("  Precision by round (share of removed images that were injected): "
    + ", ".join(f"{h / per_round * 100:.1f}%" for h in prec)
    + f"   [random chance: {n_inj / n0 * 100:.1f}%]")
log(f"  Injected images found after all rounds: {prec.sum():.0f} of {n_inj} "
    f"({prec.sum() / n_inj * 100:.0f}%)")
log(f"  Injected share left in the training set: "
    f"{np.mean([r['inj_left'] for r in results]) * 100:.2f}% (started at {n_inj / n0 * 100:.2f}%)")
log("\nThe ceiling is the no-injected-anomaly reference; it is not an additional cleaning method.")


## 10. Selected anomaly images
Two image-only galleries show the 30 highest- and 30 lowest-scoring images selected for removal during seed 1.

In [ ]:
from IPython.display import Image as NotebookImage, display

DISPLAY_SAMPLE_SIZE = 30
candidates = results[0].get("selected_anomalies", [])
if len(candidates) < 2:
    raise RuntimeError("Not enough selected images to show high- and low-score examples.")

sample_size = min(DISPLAY_SAMPLE_SIZE, len(candidates) // 2)
by_score = sorted(candidates, key=lambda item: item["score"])
low_score_sample = by_score[:sample_size]
high_score_sample = by_score[-sample_size:][::-1]

def show_selected_image_sample(items, title, filename):
    cols = 6
    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 2.0), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, item in zip(axes.flat, items):
        ax.imshow(item["image"], cmap="gray", vmin=0, vmax=1)
        truth = "injected" if item["injected"] else "normal"
        ax.set_title(
            f"digit {item['digit']} ({truth})\nscore {item['score']:.3f} | round {item['round']}",
            fontsize=7,
        )
    fig.suptitle(title + " — seed 1; score is relative suspicion rank, not a probability")
    fig.tight_layout()
    image_path = OUTPUT_DIR / f"{filename}_{RUN_ID}.png"
    image_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(image_path, dpi=140, bbox_inches="tight")
    plt.close(fig)
    display(NotebookImage(filename=str(image_path)))
    log(f"Displayed and saved {len(items)} selected images to: {image_path}")

show_selected_image_sample(
    high_score_sample,
    f"Top {sample_size} highest-scoring selected anomalies",
    "high",
)
show_selected_image_sample(
    low_score_sample,
    f"Bottom {sample_size} lowest-scoring selected anomalies",
    "low",
)

## 11. Save results

In [ ]:
out_path = OUTPUT_DIR / f"{RESULT_STEM}.txt"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text("\n".join(LOG_LINES), encoding="utf-8")            # everything passed to log() goes in the file
print(f"Saved results to: {out_path}")

## Notes

- The ceiling is the no-injected-anomaly reference for iterative cleaning.
- AUROC near 50% means the selected score separates anomalies poorly.
- Results from few seeds can vary. Increase `NUM_LATENT_SPACES` for a steadier estimate.
- `SCORE_MODE` selects latent distance, reconstruction error, or both.